# Archive Intraday Data

Notebook-only archive helper.
There is no current `.py` runtime twin for this file, so this notebook is the manual version.

In [ ]:
import time
from datetime import datetime, timedelta, timezone

from sqlalchemy import BigInteger, Column, DateTime, Float, MetaData, String, Table, UniqueConstraint, text

from backfill_config import ARCHIVE_RETENTION_DAYS, MAIN_TABLE_RETENTION_DAYS
from backfill_utils import ensure_schema, get_engine

In [ ]:
def ensure_archive_schema(engine) -> None:
    ensure_schema(engine)

    metadata = MetaData()
    Table(
        "stock_intraday_archive",
        metadata,
        Column("id", BigInteger, primary_key=True, autoincrement=True),
        Column("symbol", String(255), nullable=False),
        Column("data_type", String(255), nullable=False),
        Column("ts", BigInteger, nullable=False),
        Column("open", Float),
        Column("high", Float),
        Column("low", Float),
        Column("close", Float),
        Column("volume", BigInteger),
        Column("date_update", DateTime),
        UniqueConstraint("symbol", "data_type", "ts", name="uq_symbol_datatype_ts_archive"),
    )

    metadata.create_all(engine)


def _ensure_tables():
    engine = get_engine()
    ensure_archive_schema(engine)
    return engine

In [ ]:
def archive_old_intraday(batch_limit: int | None = None, sleep_sec: float = 0.0):
    engine = _ensure_tables()
    now_utc = datetime.now(timezone.utc)

    with engine.begin() as conn:
        for interval, keep_days in MAIN_TABLE_RETENTION_DAYS.items():
            if interval == "1d" or keep_days >= 99999:
                continue

            cutoff_ts = int((now_utc - timedelta(days=int(keep_days))).timestamp())

            if batch_limit is None:
                insert_sql = """
                INSERT INTO stock_intraday_archive (symbol, data_type, ts, open, high, low, close, volume)
                SELECT symbol, data_type, ts, open, high, low, close, volume
                FROM stock_ohlc
                WHERE data_type = :interval AND ts < :cutoff_ts
                ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts_archive DO NOTHING
                """
                conn.execute(text(insert_sql), {"interval": interval, "cutoff_ts": cutoff_ts})

                delete_sql = """
                DELETE FROM stock_ohlc
                WHERE data_type = :interval AND ts < :cutoff_ts
                """
                deleted = conn.execute(text(delete_sql), {"interval": interval, "cutoff_ts": cutoff_ts}).rowcount
                print(f"[ARCHIVE MOVE] interval={interval} deleted_from_main={deleted} cutoff_ts={cutoff_ts}")
            else:
                while True:
                    move_sql = """
                    WITH moved AS (
                        SELECT symbol, data_type, ts, open, high, low, close, volume
                        FROM stock_ohlc
                        WHERE data_type = :interval AND ts < :cutoff_ts
                        ORDER BY ts
                        LIMIT :limit
                    ), ins AS (
                        INSERT INTO stock_intraday_archive (symbol, data_type, ts, open, high, low, close, volume)
                        SELECT symbol, data_type, ts, open, high, low, close, volume
                        FROM moved
                        ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts_archive DO NOTHING
                        RETURNING 1
                    )
                    DELETE FROM stock_ohlc o
                    USING moved m
                    WHERE o.symbol = m.symbol AND o.data_type = m.data_type AND o.ts = m.ts
                    RETURNING 1
                    """
                    res = conn.execute(text(move_sql), {"interval": interval, "cutoff_ts": cutoff_ts, "limit": int(batch_limit)})
                    deleted = res.rowcount
                    if not deleted:
                        break
                    print(f"[ARCHIVE MOVE] interval={interval} deleted_batch={deleted} cutoff_ts={cutoff_ts}")
                    if sleep_sec:
                        time.sleep(float(sleep_sec))

    engine.dispose()


def cleanup_archive_expired(batch_limit: int | None = None, sleep_sec: float = 0.0):
    engine = _ensure_tables()
    now_utc = datetime.now(timezone.utc)

    with engine.begin() as conn:
        for interval, keep_days in ARCHIVE_RETENTION_DAYS.items():
            if interval == "1d" or keep_days >= 99999:
                continue

            cutoff_ts = int((now_utc - timedelta(days=int(keep_days))).timestamp())

            if batch_limit is None:
                delete_sql = """
                DELETE FROM stock_intraday_archive
                WHERE data_type = :interval AND ts < :cutoff_ts
                """
                deleted = conn.execute(text(delete_sql), {"interval": interval, "cutoff_ts": cutoff_ts}).rowcount
                print(f"[ARCHIVE CLEANUP] interval={interval} deleted_from_archive={deleted} cutoff_ts={cutoff_ts}")
            else:
                while True:
                    delete_sql = """
                    WITH doomed AS (
                        SELECT id
                        FROM stock_intraday_archive
                        WHERE data_type = :interval AND ts < :cutoff_ts
                        ORDER BY ts
                        LIMIT :limit
                    )
                    DELETE FROM stock_intraday_archive a
                    USING doomed d
                    WHERE a.id = d.id
                    RETURNING 1
                    """
                    res = conn.execute(text(delete_sql), {"interval": interval, "cutoff_ts": cutoff_ts, "limit": int(batch_limit)})
                    deleted = res.rowcount
                    if not deleted:
                        break
                    print(f"[ARCHIVE CLEANUP] interval={interval} deleted_batch={deleted} cutoff_ts={cutoff_ts}")
                    if sleep_sec:
                        time.sleep(float(sleep_sec))

    engine.dispose()

In [ ]:
# Examples:
# archive_old_intraday(batch_limit=50000, sleep_sec=0.1)
# cleanup_archive_expired(batch_limit=50000, sleep_sec=0.1)

archive_old_intraday(batch_limit=50000, sleep_sec=0.1)